In [0]:
%run ../_shared/00_project_config

In [0]:
import csv
import io
import os
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide league mappings, seasons, and raw storage paths are loaded from
# 00_project_config.
#
# This notebook is responsible only for downloading raw football-data.co.uk
# CSV files into the landing area. No cleaning or transformation is performed
# here.

BASE_URL = "https://www.football-data.co.uk/mmz4281"

REQUEST_TIMEOUT_SECONDS = 30

# Historical files are treated as immutable during the current project phase.
#
# Existing files are therefore reused rather than downloaded again.
#
# When incremental/current-season ingestion is introduced later, this behavior
# should be revisited so an active season can be refreshed safely.
SKIP_EXISTING_FILES = True


# -----------------------------------------------------------------------------
# HTTP session
# -----------------------------------------------------------------------------
# Reuse one HTTP connection across downloads and retry temporary server/network
# failures automatically.
#
# Retries are intentionally limited to transient server-side responses.
# Client errors such as 404 should fail normally rather than being retried.

retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=1.5,
    status_forcelist=[
        500,
        502,
        503,
        504,
    ],
    allowed_methods=frozenset({"GET"}),
    respect_retry_after_header=True,
)

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session = requests.Session()

session.mount(
    "https://",
    adapter,
)


# -----------------------------------------------------------------------------
# Response validation
# -----------------------------------------------------------------------------

def validate_csv_response(
    content: bytes,
    *,
    url: str,
) -> None:
    """
    Validate that the downloaded payload looks like a football-data.co.uk CSV.

    This protects the raw landing area from accidentally storing:
      - empty responses
      - HTML error pages
      - malformed/non-CSV responses

    Validation remains intentionally lightweight because Bronze should preserve
    the original source file rather than impose transformation logic here.
    """

    if not content:
        raise ValueError(
            f"Empty response received from: {url}"
        )

    # football-data.co.uk CSV files are text files. utf-8-sig safely handles
    # an optional UTF-8 BOM if one is present.
    try:
        text = content.decode(
            "utf-8-sig"
        )
    except UnicodeDecodeError as exc:
        raise ValueError(
            f"Response is not valid UTF-8 text: {url}"
        ) from exc

    stripped_text = text.lstrip()

    # An upstream web server may occasionally return an HTML error document
    # with HTTP 200. Do not persist such content as a .csv file.
    if (
        stripped_text.lower().startswith("<!doctype html")
        or stripped_text.lower().startswith("<html")
    ):
        raise ValueError(
            f"Received HTML instead of CSV data: {url}"
        )

    reader = csv.DictReader(
        io.StringIO(text)
    )

    fieldnames = reader.fieldnames

    if not fieldnames:
        raise ValueError(
            f"CSV response contains no header row: {url}"
        )

    required_columns = {
        "Div",
        "Date",
        "HomeTeam",
        "AwayTeam",
    }

    missing_columns = (
        required_columns
        - set(fieldnames)
    )

    if missing_columns:
        raise ValueError(
            "Downloaded CSV does not contain the expected "
            f"football-data.co.uk columns. "
            f"Missing={sorted(missing_columns)}, url={url}"
        )

    # Make sure the file contains at least one data row.
    try:
        first_row = next(reader)
    except StopIteration as exc:
        raise ValueError(
            f"CSV contains a header but no match rows: {url}"
        ) from exc

    if not first_row:
        raise ValueError(
            f"CSV contains an invalid first data row: {url}"
        )


# -----------------------------------------------------------------------------
# Atomic file writing
# -----------------------------------------------------------------------------

def write_bytes_atomic(
    content: bytes,
    file_path: Path,
) -> None:
    """
    Write raw bytes through a temporary file and replace the final path only
    after the complete write succeeds.

    This prevents an interrupted notebook run from leaving a partially written
    CSV at the final landing path.
    """

    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_suffix(
        file_path.suffix + ".tmp"
    )

    try:
        with temporary_path.open(
            "wb"
        ) as file:
            file.write(
                content
            )

        os.replace(
            temporary_path,
            file_path,
        )

    except Exception:

        if temporary_path.exists():
            temporary_path.unlink()

        raise


# -----------------------------------------------------------------------------
# Ingestion state
# -----------------------------------------------------------------------------

downloaded_count = 0
skipped_count = 0
failed_downloads: list[str] = []


# -----------------------------------------------------------------------------
# Download football-data.co.uk CSV files
# -----------------------------------------------------------------------------
# Processing order is deterministic:
#
#   season -> league
#
# One raw source file is stored for each league-season combination.

try:

    for season in FOOTBALL_DATA_SEASONS:

        print(
            "\n"
            "============================================================"
        )

        print(
            f"Processing football-data.co.uk season {season}"
        )

        print(
            "============================================================"
        )

        for league_code, league_name in FOOTBALL_DATA_LEAGUES.items():

            url = (
                f"{BASE_URL}/"
                f"{season}/"
                f"{league_code}.csv"
            )

            file_path = Path(
                FOOTBALL_DATA_RAW_ROOT,
                season,
                f"{season}_{league_name}.csv",
            )


            # -----------------------------------------------------------------
            # Reuse existing raw file
            # -----------------------------------------------------------------

            if (
                SKIP_EXISTING_FILES
                and file_path.exists()
            ):

                skipped_count += 1

                print(
                    f"SKIP      | "
                    f"{season} | "
                    f"{league_name} | "
                    f"file already exists"
                )

                continue


            # -----------------------------------------------------------------
            # Download source file
            # -----------------------------------------------------------------

            print(
                f"DOWNLOAD  | "
                f"{season} | "
                f"{league_name}"
            )

            try:

                response = session.get(
                    url,
                    timeout=REQUEST_TIMEOUT_SECONDS,
                )

                response.raise_for_status()


                # -------------------------------------------------------------
                # Validate response before persistence
                # -------------------------------------------------------------

                validate_csv_response(
                    response.content,
                    url=url,
                )


                # -------------------------------------------------------------
                # Persist original source bytes
                # -------------------------------------------------------------
                # The downloaded file is written without changing its content.
                # Cleaning, casting, and standardization belong in Bronze /
                # Silver processing rather than raw ingestion.

                write_bytes_atomic(
                    response.content,
                    file_path,
                )

                downloaded_count += 1

                print(
                    f"SUCCESS   | "
                    f"{season} | "
                    f"{league_name} | "
                    f"{file_path}"
                )


            except (
                requests.RequestException,
                ValueError,
                OSError,
            ) as exc:

                failure_message = (
                    f"{season} | "
                    f"{league_name} | "
                    f"{url} | "
                    f"{type(exc).__name__}: {exc}"
                )

                failed_downloads.append(
                    failure_message
                )

                print(
                    f"FAILED    | {failure_message}"
                )

finally:

    session.close()


# -----------------------------------------------------------------------------
# Run summary
# -----------------------------------------------------------------------------

expected_files = (
    len(FOOTBALL_DATA_SEASONS)
    * len(FOOTBALL_DATA_LEAGUES)
)

print(
    "\n"
    "============================================================"
)

print(
    "football-data.co.uk ingestion summary"
)

print(
    "============================================================"
)

print(
    f"Expected league-season files: {expected_files}"
)

print(
    f"Downloaded this run: {downloaded_count}"
)

print(
    f"Existing files reused: {skipped_count}"
)

print(
    f"Failed downloads: {len(failed_downloads)}"
)


# -----------------------------------------------------------------------------
# Fail the notebook if any expected download failed
# -----------------------------------------------------------------------------
# Individual downloads are attempted independently so one temporary failure
# does not prevent other league-season files from being retrieved.
#
# The notebook still fails at the end when any extraction failed. This is
# important for future orchestration because a Databricks Job must not report
# SUCCESS when the raw dataset is incomplete.

if failed_downloads:

    failure_details = "\n".join(
        f"  - {failure}"
        for failure in failed_downloads
    )

    raise RuntimeError(
        "football-data.co.uk ingestion completed with failures:\n"
        f"{failure_details}"
    )


print(
    "\nfootball-data.co.uk ingestion completed successfully."
)